# robo_v4_orange.ipynb
- ROBOT_ID: D
- IP: 194.47.156.43
- Robot flag: Orange

### Changes from v3:
- Obstacle avoidance removed
- STOP_DISTANCE increased to 40cm

In [ ]:
import cv2
import numpy as np
import requests
import threading
import time
import json
import urllib.parse
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot
from http.server import BaseHTTPRequestHandler, HTTPServer

# --- Config ---
ROBOT_ID      = 'D'
FOCAL_LENGTH  = 257.39
REAL_WIDTH    = 23              # cm - physical size of red box
STOP_DISTANCE = 40              # cm - stop when this close to any target
BASE_SPEED    = 0.2
LOOP_DELAY    = 0.15            # seconds between frames

# Flag colors per robot
ROBOT_COLORS = {
    'A': 'blue',
    'B': 'yellow',
    'C': 'purple',
    'D': 'orange',
}

MY_COLOR = ROBOT_COLORS[ROBOT_ID]

In [2]:
# --- Hardware init ---
def bgr8_to_jpeg(value):
    return cv2.imencode('.jpg', value)[1].tobytes()

camera = Camera.instance(width=720, height=480)
robot  = Robot()

In [3]:
# --- Shared state ---
status_lock = threading.Lock()

latest_detection = {
    "robot_id": ROBOT_ID,
    "red_box":  {"detected": False, "distance": None},
    "robot_A":  {"detected": False, "distance": None},
    "robot_B":  {"detected": False, "distance": None},
    "robot_C":  {"detected": False, "distance": None},
    "robot_D":  {"detected": False, "distance": None},
}

global_best_id = None

In [4]:
# --- HTTP server ---

class CameraHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path.startswith('/camera'):
            self.send_response(200)
            self.send_header('Content-type', 'image/jpeg')
            self.end_headers()
            self.wfile.write(bgr8_to_jpeg(camera.value))

        elif self.path.startswith('/status'):
            with status_lock:
                payload = json.dumps(latest_detection).encode()
            self.send_response(200)
            self.send_header('Content-type', 'application/json')
            self.end_headers()
            self.wfile.write(payload)

        elif self.path.startswith('/set_target'):
            global global_best_id
            query  = urllib.parse.urlparse(self.path).query
            params = urllib.parse.parse_qs(query)
            global_best_id = params.get('robot_id', [None])[0]
            self._ok(b'Target set')

        elif self.path.startswith('/start'):
            threading.Thread(target=main_loop, daemon=True).start()
            self._ok(b'Started')

        elif self.path.startswith('/stop'):
            robot.stop()
            self._ok(b'Stopped')

        else:
            query  = urllib.parse.urlparse(self.path).query
            params = urllib.parse.parse_qs(query)

            if self.path.startswith('/set_motors'):
                left  = float(params.get('left',  [0])[0])
                right = float(params.get('right', [0])[0])
                robot.set_motors(left, right)
                self._ok(b'Motors set')

            elif self.path.startswith('/forward'):
                speed = float(params.get('speed', [0])[0])
                robot.forward(speed)
                self._ok(b'Forward')

            elif self.path.startswith('/left'):
                speed = float(params.get('speed', [0])[0])
                robot.left(speed)
                self._ok(b'Left')

            elif self.path.startswith('/right'):
                speed = float(params.get('speed', [0])[0])
                robot.right(speed)
                self._ok(b'Right')

    def _ok(self, msg):
        self.send_response(200)
        self.end_headers()
        self.wfile.write(msg)

    def log_message(self, format, *args):
        pass

def run_server():
    server = HTTPServer(('0.0.0.0', 8089), CameraHandler)
    server.serve_forever()

threading.Thread(target=run_server, daemon=True).start()
print("Robot server running on port 8089")

Robot server running on port 8089


In [5]:
# --- Multi-color detection ---

COLOR_RANGES = {
    'red':    (np.array([0,   120,  70]), np.array([10,  255, 255]),
               np.array([170, 120,  70]), np.array([180, 255, 255])),
    'blue':   (np.array([100, 150,  50]), np.array([130, 255, 255]),
               None, None),
    'orange': (np.array([11,  200, 200]), np.array([18,  255, 255]),
               None, None),
    'yellow': (np.array([20,  100, 100]), np.array([40,  255, 255]),
               None, None),
    'purple': (np.array([130,  50,  50]), np.array([160, 255, 255]),
               None, None),
}

def detect_color(frame, color):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower1, upper1, lower2, upper2 = COLOR_RANGES[color]
    mask = cv2.inRange(hsv, lower1, upper1)
    if lower2 is not None:
        mask = mask + cv2.inRange(hsv, lower2, upper2)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        if cv2.contourArea(largest) > 50:
            x, y, w, h = cv2.boundingRect(largest)
            return {"x": x, "y": y, "w": w, "h": h}
    return None

def scan_frame(frame):
    results = {"red_box": detect_color(frame, 'red')}
    for robot_id, color in ROBOT_COLORS.items():
        if robot_id != ROBOT_ID:
            results[f'robot_{robot_id}'] = detect_color(frame, color)
    return results

def estimate_distance(pixel_width):
    return (REAL_WIDTH * FOCAL_LENGTH) / pixel_width

def calculate_steering(frame_width, x, w):
    return (x + w / 2) - (frame_width / 2)

In [6]:
# --- Movement ---

def steer_toward(error_x):
    normalized = max(-1.0, min(1.0, error_x / 240.0))
    forward    = BASE_SPEED * (1 - abs(normalized) * 0.5)
    turn       = normalized * 0.15
    robot.set_motors(max(0.05, forward + turn),
                     max(0.05, forward - turn))

def spin_and_search():
    robot.set_motors(0.12, -0.12)

In [7]:
# --- Priority logic ---

def decide_and_act(scan_results, frame_width):

    # Priority 1 — red box visible
    box = scan_results.get('red_box')
    if box:
        distance = estimate_distance((box['w'] + box['h']) / 2)
        if distance < STOP_DISTANCE:
            robot.stop()
            return f"STOPPED — reached box at {distance:.1f}cm"
        error_x = calculate_steering(frame_width, box['x'], box['w'])
        steer_toward(error_x)
        return f"P1 — steering toward box, distance={distance:.1f}cm"

    # Priority 2 — global best robot's flag visible
    if global_best_id and global_best_id != ROBOT_ID:
        flag = scan_results.get(f'robot_{global_best_id}')
        if flag:
            distance = estimate_distance((flag['w'] + flag['h']) / 2)
            if distance < STOP_DISTANCE:
                robot.stop()
                return f"STOPPED — reached Robot {global_best_id} at {distance:.1f}cm"
            error_x = calculate_steering(frame_width, flag['x'], flag['w'])
            steer_toward(error_x)
            return f"P2 — following Robot {global_best_id}, distance={distance:.1f}cm"

    # Priority 3 — any other robot flag visible
    for robot_id in ROBOT_COLORS:
        if robot_id == ROBOT_ID:
            continue
        flag = scan_results.get(f'robot_{robot_id}')
        if flag:
            distance = estimate_distance((flag['w'] + flag['h']) / 2)
            if distance < STOP_DISTANCE:
                robot.stop()
                return f"STOPPED — reached Robot {robot_id} at {distance:.1f}cm"
            error_x = calculate_steering(frame_width, flag['x'], flag['w'])
            steer_toward(error_x)
            return f"P3 — following Robot {robot_id}, distance={distance:.1f}cm"

    # Priority 4 — nothing visible
    spin_and_search()
    return "P4 — spinning to search" 

In [8]:
# --- Update shared status ---

def update_status(scan_results):
    with status_lock:
        box = scan_results.get('red_box')
        if box:
            dist = estimate_distance((box['w'] + box['h']) / 2)
            latest_detection['red_box'] = {"detected": True, "distance": round(dist, 2)}
        else:
            latest_detection['red_box'] = {"detected": False, "distance": None}

        for robot_id in ROBOT_COLORS:
            if robot_id == ROBOT_ID:
                continue
            key  = f'robot_{robot_id}'
            flag = scan_results.get(key)
            if flag:
                dist = estimate_distance((flag['w'] + flag['h']) / 2)
                latest_detection[key] = {"detected": True, "distance": round(dist, 2)}
            else:
                latest_detection[key] = {"detected": False, "distance": None}

In [9]:
# --- Livestream ---
image_widget = widgets.Image(format='jpeg', width=480, height=480)
display(image_widget)

def update_stream():
    while True:
        image_widget.value = bgr8_to_jpeg(camera.value)
        time.sleep(0.1)

threading.Thread(target=update_stream, daemon=True).start()
print("Livestream running above")

Image(value=b'', format='jpeg', height='480', width='480')

Livestream running above


In [10]:
# --- Main loop ---

def main_loop():
    try:
        while True:
            frame        = camera.value.copy()
            scan_results = scan_frame(frame)
            update_status(scan_results)
            action = decide_and_act(scan_results, frame.shape[1])
            print(f"[{ROBOT_ID}] {action}")
            time.sleep(LOOP_DELAY)

    except KeyboardInterrupt:
        robot.stop()
        print("Stopped.")

# main_loop()  # uncomment to start manually
# or send GET /start from laptop

[D] P1 — steering toward box, distance=123.3cm
[D] P1 — steering toward box, distance=124.6cm
[D] P1 — steering toward box, distance=120.8cm
[D] P1 — steering toward box, distance=119.6cm
[D] P1 — steering toward box, distance=115.0cm
[D] P1 — steering toward box, distance=112.8cm
[D] P1 — steering toward box, distance=108.6cm
[D] P1 — steering toward box, distance=103.9cm
[D] P1 — steering toward box, distance=98.7cm
[D] P1 — steering toward box, distance=95.5cm
[D] P1 — steering toward box, distance=91.1cm
[D] P1 — steering toward box, distance=87.1cm
[D] P1 — steering toward box, distance=82.8cm
[D] P1 — steering toward box, distance=80.0cm
[D] P1 — steering toward box, distance=74.9cm
[D] P1 — steering toward box, distance=71.8cm
[D] P1 — steering toward box, distance=69.2cm
[D] P1 — steering toward box, distance=65.4cm
[D] P1 — steering toward box, distance=61.0cm
[D] P1 — steering toward box, distance=56.7cm
[D] P1 — steering toward box, distance=53.3cm
[D] P1 — steering toward b